# Laboratorio 8 — MM3014 Teoría de Probabilidades
## Simulación Monte Carlo: Álbum Panini con Presupuesto e Intercambio

**Universidad del Valle de Guatemala**  
**Autores:** Angel Sanabria (24725) · Derek Coronado (24732)  
**Semilla global:** `2026`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Parámetros globales
N      = 100    # Estampas totales
S      = 7      # Estampas por sobre
R      = 10_000 # Simulaciones
SEED   = 2026

# Estilo
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f8f9fa',
    'axes.grid'        : True,
    'grid.alpha'       : 0.35,
    'font.size'        : 11,
})

print(f'Parámetros: N={N}, S={S}, R={R:,}, seed={SEED}')

---
## Etapa 3 — Incorporación del presupuesto y costo

### Parámetros adicionales
| Parámetro | Valor |
|---|---|
| Precio por sobre | Q 9.50 |
| Presupuesto total | Q 1 000 |
| Simulaciones | 10 000 |

In [ ]:
# ── Parámetros Etapa 3 ──────────────────────────────────────────────────────
PRECIO_SOBRE = 9.50
PRESUPUESTO  = 1_000.0

MAX_SOBRES_POSIBLES = int(PRESUPUESTO // PRECIO_SOBRE)
print(f'Máximo de sobres comprables con Q{PRESUPUESTO:.0f}: {MAX_SOBRES_POSIBLES} sobres')
print(f'Costo total máximo : Q{MAX_SOBRES_POSIBLES * PRECIO_SOBRE:.2f}')
print(f'Vuelto             : Q{PRESUPUESTO - MAX_SOBRES_POSIBLES * PRECIO_SOBRE:.2f}')

### Simulación principal — sobres sueltos con presupuesto

In [ ]:
def simular_con_presupuesto(N, S, precio, presupuesto, R, seed):
    """
    Simula R veces la compra secuencial de sobres con restricción de presupuesto.

    Returns
    -------
    completado    : ndarray(bool)  — si se completó el álbum
    sobres_comp   : ndarray(int)   — sobres comprados por simulación
    distintas_fail: ndarray(int)   — estampas distintas en simulaciones fallidas
    """
    rng = np.random.default_rng(seed)

    completado     = np.zeros(R, dtype=bool)
    sobres_comp    = np.zeros(R, dtype=int)
    distintas_fin  = np.zeros(R, dtype=int)

    for sim in range(R):
        coleccion = np.zeros(N, dtype=bool)
        gasto     = 0.0
        sobres    = 0

        while (gasto + precio) <= presupuesto and not coleccion.all():
            sobre   = rng.choice(N, size=S, replace=False)
            coleccion[sobre] = True
            gasto  += precio
            sobres += 1

        completado[sim]    = coleccion.all()
        sobres_comp[sim]   = sobres
        distintas_fin[sim] = coleccion.sum()

    distintas_fail = distintas_fin[~completado]
    return completado, sobres_comp, distintas_fail


completado_suelto, sobres_suelto, distintas_fail_suelto = simular_con_presupuesto(
    N, S, PRECIO_SOBRE, PRESUPUESTO, R, SEED
)

prob_exito_suelto    = completado_suelto.mean()
media_sobres_suelto  = sobres_suelto.mean()
media_dist_fail      = distintas_fail_suelto.mean() if len(distintas_fail_suelto) else float('nan')

print('\n── Resultados: sobres sueltos con Q1000 ──────────────────────')
print(f'  P(completar álbum)                    : {prob_exito_suelto:.4f}  ({100*prob_exito_suelto:.2f}%)')
print(f'  Número esperado de sobres comprados   : {media_sobres_suelto:.4f}')
print(f'  Simulaciones exitosas                 : {completado_suelto.sum():,}')
print(f'  Simulaciones fallidas                 : {(~completado_suelto).sum():,}')
print(f'  E[estampas distintas | fallo]         : {media_dist_fail:.4f}')

### Visualización — Completó vs. No completó

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

categorias = ['Completó el álbum', 'No completó']
proporciones = [prob_exito_suelto, 1 - prob_exito_suelto]
colores = ['#2ecc71', '#e74c3c']

bars = ax.bar(categorias, proporciones, color=colores, edgecolor='black',
              linewidth=1.2, width=0.5)

for bar, val in zip(bars, proporciones):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_ylim(0, 1.15)
ax.set_ylabel('Proporción de simulaciones', fontsize=12)
ax.set_title('Etapa 3 — Completar el álbum con Q 1,000\n'
             f'(N={N}, S={S}, precio=Q{PRECIO_SOBRE}, R={R:,})', fontsize=13)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig('etapa3_completado_vs_no.png', dpi=150)
plt.show()

---
### Preguntas de análisis — Etapa 3

#### Pregunta 1 — Máximo de sobres con Q 1,000 y mínimo teórico

In [ ]:
minimo_teorico_sobres = int(np.ceil(N / S))  # sin ninguna repetición

print('── Pregunta 1 ───────────────────────────────────────────────')
print(f'  Máximo sobres comprables con Q{PRESUPUESTO:.0f}   : {MAX_SOBRES_POSIBLES} sobres')
print(f'    Costo total                        : Q{MAX_SOBRES_POSIBLES * PRECIO_SOBRE:.2f}')
print()
print(f'  Mínimo teórico (ceil(N/S) = ceil({N}/{S})): {minimo_teorico_sobres} sobres')
print(f'    Estampas que entregarían           : {minimo_teorico_sobres * S} (>= N={N})')
print()
print(f'  Conclusión:')
print(f'    Con Q{PRESUPUESTO:.0f} se pueden comprar {MAX_SOBRES_POSIBLES} sobres,',
      f'muy por encima del mínimo teórico de {minimo_teorico_sobres}.')
print(f'    Aun así la probabilidad de completar es {prob_exito_suelto:.1%} porque')
print(f'    en la práctica las repetidas consumen una fracción importante de esos sobres.')

#### Pregunta 2 — Caja de 104 sobres (Q 975)

In [ ]:
SOBRES_CAJA = 104
PRECIO_CAJA = 975.0

def simular_exactamente_M_sobres(N, S, M, R, seed):
    """Compra exactamente M sobres; retorna array bool de si completó."""
    rng = np.random.default_rng(seed)
    completado = np.zeros(R, dtype=bool)
    for sim in range(R):
        coleccion = np.zeros(N, dtype=bool)
        for _ in range(M):
            sobre = rng.choice(N, size=S, replace=False)
            coleccion[sobre] = True
        completado[sim] = coleccion.all()
    return completado


completado_caja = simular_exactamente_M_sobres(N, S, SOBRES_CAJA, R, SEED)
prob_exito_caja = completado_caja.mean()

print('── Pregunta 2 — Caja de 104 sobres (Q 975) ─────────────────')
print(f'  Sobres en la caja               : {SOBRES_CAJA}')
print(f'  Costo de la caja                : Q{PRECIO_CAJA:.2f}')
print(f'  Presupuesto restante tras caja  : Q{PRESUPUESTO - PRECIO_CAJA:.2f}')
print()
print(f'  P(completar | caja)             : {prob_exito_caja:.4f}  ({100*prob_exito_caja:.2f}%)')
print(f'  P(completar | sueltos Q1000)    : {prob_exito_suelto:.4f}  ({100*prob_exito_suelto:.2f}%)')
print()
diferencia = prob_exito_caja - prob_exito_suelto
print(f'  Diferencia (caja - sueltos)     : {diferencia:+.4f}  ({100*diferencia:+.2f} pp)')
if diferencia > 0:
    print('  → La caja es MEJOR que comprar sueltos con el mismo presupuesto.')
else:
    print('  → Comprar sueltos es ligeramente mejor o igual a la caja.')

#### Pregunta 3 — Estrategia mixta: caja + sobres sueltos

In [ ]:
# Con la caja (Q975) sobran Q25.  Con Q25 se pueden comprar floor(25/9.50)=2 sobres sueltos.
PRESUPUESTO_RESTANTE_CAJA = PRESUPUESTO - PRECIO_CAJA
SOBRES_EXTRA = int(PRESUPUESTO_RESTANTE_CAJA // PRECIO_SOBRE)
SOBRES_MIXTO = SOBRES_CAJA + SOBRES_EXTRA
COSTO_MIXTO  = PRECIO_CAJA + SOBRES_EXTRA * PRECIO_SOBRE

completado_mixto = simular_exactamente_M_sobres(N, S, SOBRES_MIXTO, R, SEED)
prob_exito_mixto = completado_mixto.mean()

print('── Pregunta 3 — Estrategia mixta ────────────────────────────')
print(f'  Caja: {SOBRES_CAJA} sobres (Q{PRECIO_CAJA:.2f})')
print(f'  Presupuesto restante tras caja : Q{PRESUPUESTO_RESTANTE_CAJA:.2f}')
print(f'  Sobres sueltos adicionales     : {SOBRES_EXTRA} (Q{SOBRES_EXTRA*PRECIO_SOBRE:.2f})')
print(f'  Total sobres                   : {SOBRES_MIXTO}')
print(f'  Gasto total                    : Q{COSTO_MIXTO:.2f}  (vuelto: Q{PRESUPUESTO - COSTO_MIXTO:.2f})')
print()
print(f'  P(completar | caja sola)       : {prob_exito_caja:.4f}  ({100*prob_exito_caja:.2f}%)')
print(f'  P(completar | caja + sueltos)  : {prob_exito_mixto:.4f}  ({100*prob_exito_mixto:.2f}%)')
print(f'  P(completar | solo sueltos)    : {prob_exito_suelto:.4f}  ({100*prob_exito_suelto:.2f}%)')
print()

# Resumen comparativo
estrategias  = ['Sueltos\n(Q1,000)', 'Caja sola\n(Q975)', 'Mixta\n(Q' + f'{COSTO_MIXTO:.0f})']
probs_est    = [prob_exito_suelto, prob_exito_caja, prob_exito_mixto]
sobres_est   = [MAX_SOBRES_POSIBLES, SOBRES_CAJA, SOBRES_MIXTO]

fig, ax = plt.subplots(figsize=(8, 5))
colores_est = ['#3498db', '#e67e22', '#9b59b6']
bars = ax.bar(estrategias, probs_est, color=colores_est, edgecolor='black',
              linewidth=1.2, width=0.45)
for bar, p, sob in zip(bars, probs_est, sobres_est):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.012,
            f'{p:.1%}\n({sob} sobres)', ha='center', va='bottom',
            fontweight='bold', fontsize=11)

ax.set_ylim(0, 1.2)
ax.set_ylabel('P(completar álbum)', fontsize=12)
ax.set_title('Etapa 3 — Comparación de estrategias de compra\n'
             f'(N={N}, S={S}, R={R:,})', fontsize=13)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig('etapa3_comparacion_estrategias.png', dpi=150)
plt.show()

mejor = estrategias[np.argmax(probs_est)].replace('\n', ' ')
print(f'  Estrategia óptima dentro de Q{PRESUPUESTO:.0f}: {mejor}')

---
## Etapa 4 — Efecto del intercambio de repetidas

> **Regla:** Cada **K** estampas repetidas acumuladas se canjean por **1 estampa nueva** (de las que faltan). El canje se aplica después de cada sobre.

### Parte A — Sobres hasta completar el álbum para distintos K

In [ ]:
def simular_hasta_completar_con_canje(N, S, K, R, seed):
    """
    Simula el proceso hasta completar el álbum usando canje de K repetidas → 1 nueva.

    Returns
    -------
    sobres : ndarray(int)  — sobres necesarios por simulación
    """
    rng = np.random.default_rng(seed)
    sobres_result = np.zeros(R, dtype=int)

    for sim in range(R):
        coleccion  = np.zeros(N, dtype=bool)
        repetidas  = 0
        sobres     = 0

        while not coleccion.all():
            sobre = rng.choice(N, size=S, replace=False)
            sobres += 1

            for e in sobre:
                if coleccion[e]:
                    repetidas += 1
                else:
                    coleccion[e] = True

            # Aplicar canjes mientras haya suficientes repetidas y falten estampas
            while repetidas >= K and not coleccion.all():
                faltantes = np.where(~coleccion)[0]
                nueva = rng.choice(faltantes)
                coleccion[nueva] = True
                repetidas -= K

        sobres_result[sim] = sobres

    return sobres_result


# Caso base sin intercambio
def simular_sin_canje(N, S, R, seed):
    rng = np.random.default_rng(seed)
    sobres_result = np.zeros(R, dtype=int)
    for sim in range(R):
        coleccion = np.zeros(N, dtype=bool)
        sobres = 0
        while not coleccion.all():
            sobre = rng.choice(N, size=S, replace=False)
            coleccion[sobre] = True
            sobres += 1
        sobres_result[sim] = sobres
    return sobres_result


K_valores = [1, 2, 5, 10]

resultados_K = {}
sobres_base = simular_sin_canje(N, S, R, SEED)
media_base  = sobres_base.mean()

print(f'Caso base (sin canje): media = {media_base:.4f} sobres, std = {sobres_base.std():.4f}')
print()

for K in K_valores:
    sobres_K = simular_hasta_completar_con_canje(N, S, K, R, SEED)
    media_K  = sobres_K.mean()
    std_K    = sobres_K.std()
    reduccion = 100 * (media_base - media_K) / media_base
    resultados_K[K] = {
        'sobres' : sobres_K,
        'media'  : media_K,
        'std'    : std_K,
        'reduccion': reduccion
    }
    print(f'K={K:2d} → media={media_K:.4f} sobres, std={std_K:.4f}, '
          f'reducción={reduccion:.2f}%')

### Histogramas superpuestos — distribución de sobres por K

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

colores_K = {1: '#e74c3c', 2: '#e67e22', 5: '#3498db', 10: '#2ecc71'}
bins = np.arange(10, 120, 2)

ax.hist(sobres_base, bins=bins, density=True, alpha=0.4,
        color='#7f8c8d', label=f'Sin canje  (μ={media_base:.1f})', edgecolor='none')

for K in K_valores:
    d = resultados_K[K]
    ax.hist(d['sobres'], bins=bins, density=True, alpha=0.55,
            color=colores_K[K],
            label=f'K={K}  (μ={d["media"]:.1f}, -{d["reduccion"]:.1f}%)',
            edgecolor='none')

ax.set_xlabel('Sobres necesarios para completar el álbum', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title('Etapa 4A — Distribución de sobres según tasa de intercambio K\n'
             f'(N={N}, S={S}, R={R:,})', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('etapa4a_histogramas_K.png', dpi=150)
plt.show()

### Tabla de resumen — Parte A

In [ ]:
print('┌───────────────┬───────────┬───────────┬──────────────┬──────────────────┐')
print('│ Configuración │   Media   │   Desv.   │  Reducción   │  Ahorro (Q9.50)  │')
print('├───────────────┼───────────┼───────────┼──────────────┼──────────────────┤')
print(f'│ Sin canje     │ {media_base:9.4f} │ {sobres_base.std():9.4f} │     —        │       —          │')
for K in K_valores:
    d = resultados_K[K]
    ahorro_sobres = media_base - d['media']
    ahorro_Q      = ahorro_sobres * PRECIO_SOBRE
    print(f'│ K = {K:<10d}│ {d["media"]:9.4f} │ {d["std"]:9.4f} │ {d["reduccion"]:9.2f}%  │  Q{ahorro_Q:11.2f}   │')
print('└───────────────┴───────────┴───────────┴──────────────┴──────────────────┘')

---
### Parte B — P(completar | M sobres) para distintos K

In [ ]:
def simular_M_sobres_con_canje(N, S, K, M, R, seed):
    """Compra exactamente M sobres con canje K; retorna proporción de éxitos."""
    rng = np.random.default_rng(seed)
    exitos = 0
    for _ in range(R):
        coleccion = np.zeros(N, dtype=bool)
        repetidas = 0
        for _ in range(M):
            sobre = rng.choice(N, size=S, replace=False)
            for e in sobre:
                if coleccion[e]:
                    repetidas += 1
                else:
                    coleccion[e] = True
            while repetidas >= K and not coleccion.all():
                faltantes = np.where(~coleccion)[0]
                nueva = rng.choice(faltantes)
                coleccion[nueva] = True
                repetidas -= K
        if coleccion.all():
            exitos += 1
    return exitos / R


def simular_M_sobres_sin_canje(N, S, M, R, seed):
    """Compra exactamente M sobres sin canje; retorna proporción de éxitos."""
    rng = np.random.default_rng(seed)
    exitos = 0
    for _ in range(R):
        coleccion = np.zeros(N, dtype=bool)
        for _ in range(M):
            sobre = rng.choice(N, size=S, replace=False)
            coleccion[sobre] = True
        if coleccion.all():
            exitos += 1
    return exitos / R


M_valores = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
R_B = 10_000

# Curva sin canje
prob_base_curva = []
for M in M_valores:
    prob_base_curva.append(simular_M_sobres_sin_canje(N, S, M, R_B, SEED))
prob_base_curva = np.array(prob_base_curva)

# Curvas por K
probs_K_curva = {}
for K in K_valores:
    probs = []
    for M in M_valores:
        probs.append(simular_M_sobres_con_canje(N, S, K, M, R_B, SEED))
    probs_K_curva[K] = np.array(probs)
    print(f'K={K} listo → P@M=45: {probs_K_curva[K][M_valores.index(45)]:.4f}')

### Gráfica de líneas — P(completar) vs M

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(M_valores, prob_base_curva, 'o--', color='#7f8c8d',
        linewidth=2, markersize=6, label='Sin canje')

for K in K_valores:
    ax.plot(M_valores, probs_K_curva[K], 'o-',
            color=colores_K[K], linewidth=2, markersize=6, label=f'K={K}')

for umbral, estilo, color in [(0.50, '--', '#c0392b'), (0.75, ':', '#8e44ad'), (0.90, '-.', '#2980b9')]:
    ax.axhline(umbral, linestyle=estilo, color=color, alpha=0.65, linewidth=1.5,
               label=f'P = {umbral:.0%}')

ax.set_xlabel('Número de sobres comprados (M)', fontsize=12)
ax.set_ylabel('P(completar álbum)', fontsize=12)
ax.set_title('Etapa 4B — Probabilidad de éxito vs. sobres para distintos K\n'
             f'(N={N}, S={S}, R={R_B:,})', fontsize=13)
ax.set_xticks(M_valores)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=10, ncol=2)
plt.tight_layout()
plt.savefig('etapa4b_prob_vs_M.png', dpi=150)
plt.show()

### Umbrales 50%, 75% y 90% por K

In [ ]:
def umbral_M(M_vals, probs, nivel):
    """Retorna el primer M donde la probabilidad supera `nivel`."""
    for M, p in zip(M_vals, probs):
        if p >= nivel:
            return M
    return f'>{ M_vals[-1]}'

print('Sobres necesarios para alcanzar cada umbral de probabilidad:')
print(f'{"Configuración":18s} {"≥50%":>10s} {"≥75%":>10s} {"≥90%":>10s}')
print('-' * 52)
print(f'{"Sin canje":18s}',
      f'{umbral_M(M_valores, prob_base_curva, 0.50):>10}',
      f'{umbral_M(M_valores, prob_base_curva, 0.75):>10}',
      f'{umbral_M(M_valores, prob_base_curva, 0.90):>10}')
for K in K_valores:
    p = probs_K_curva[K]
    print(f'{f"K={K}":18s}',
          f'{umbral_M(M_valores, p, 0.50):>10}',
          f'{umbral_M(M_valores, p, 0.75):>10}',
          f'{umbral_M(M_valores, p, 0.90):>10}')

---
### Preguntas de análisis — Etapa 4

#### Pregunta 1 — ¿Cómo afecta la disminución de K?

In [ ]:
medias = [resultados_K[K]['media'] for K in K_valores]
reduc  = [resultados_K[K]['reduccion'] for K in K_valores]

print('── Pregunta 1 ───────────────────────────────────────────────')
print(f'  Sin canje : media = {media_base:.2f} sobres')
for K, m, r in zip(K_valores, medias, reduc):
    print(f'  K={K:<2d}       : media = {m:.2f} sobres  (reducción {r:.2f}%)')

# Saltos entre K consecutivos
print()
print('  Incremento de mejora al reducir K (saltos):')
prev = media_base
for K, m in zip(K_valores, medias):
    print(f'    Sin canje → K={K}: ahorro adicional {prev - m:.2f} sobres')
    prev = m

print()
print('  Conclusión: La relación no es lineal. Los mayores saltos ocurren')
print('  al pasar a K pequeños (1 y 2). A medida que K decrece, los canjes')
print('  se vuelven más frecuentes, acelerando la colección no linealmente.')

#### Pregunta 2 — Ahorro en sobres y quetzales para K = 2

In [ ]:
K2 = resultados_K[2]
ahorro_sobres_K2 = media_base - K2['media']
ahorro_Q_K2 = ahorro_sobres_K2 * PRECIO_SOBRE

print('── Pregunta 2 — Ahorro con K=2 ─────────────────────────────')
print(f'  Media sin canje    : {media_base:.4f} sobres')
print(f'  Media con K=2      : {K2["media"]:.4f} sobres')
print(f'  Ahorro en sobres   : {ahorro_sobres_K2:.4f} sobres')
print(f'  Ahorro en quetzales: Q{ahorro_Q_K2:.2f}  (a Q{PRECIO_SOBRE}/sobre)')

#### Pregunta 3 — Variación de probabilidad para M = 45

In [ ]:
idx_45 = M_valores.index(45)

p_base_45 = prob_base_curva[idx_45]
p_K10_45  = probs_K_curva[10][idx_45]
p_K5_45   = probs_K_curva[5][idx_45]
p_K1_45   = probs_K_curva[1][idx_45]

print('── Pregunta 3 — M = 45 sobres ───────────────────────────────')
print(f'  P(completar | sin canje, M=45) : {p_base_45:.4f}')
print(f'  P(completar | K=10,     M=45) : {p_K10_45:.4f}')
print(f'  P(completar | K=5,      M=45) : {p_K5_45:.4f}')
print(f'  P(completar | K=1,      M=45) : {p_K1_45:.4f}')
print()
print(f'  Mejora al pasar K=10 → K=5   : +{p_K5_45 - p_K10_45:.4f} ({100*(p_K5_45 - p_K10_45):.2f} pp)')
print(f'  Mejora al pasar K=5  → K=1   : +{p_K1_45 - p_K5_45:.4f} ({100*(p_K1_45 - p_K5_45):.2f} pp)')

#### Pregunta 4 — ¿Existe un K con rendimiento marginal decreciente?

In [ ]:
K_extra = [1, 2, 3, 4, 5, 7, 10]
medias_extra = []
for K in K_extra:
    s = simular_hasta_completar_con_canje(N, S, K, 5_000, SEED)
    medias_extra.append(s.mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_extra, medias_extra, 'o-', color='#8e44ad', linewidth=2, markersize=8)
ax.axhline(media_base, linestyle='--', color='#7f8c8d', label=f'Sin canje (μ={media_base:.1f})')
ax.set_xlabel('K (repetidas necesarias para 1 canje)', fontsize=12)
ax.set_ylabel('Media de sobres para completar', fontsize=12)
ax.set_title('Rendimiento marginal del intercambio en función de K\n'
             f'(N={N}, S={S}, R=5,000)', fontsize=13)
ax.set_xticks(K_extra)
ax.legend()
plt.tight_layout()
plt.savefig('etapa4_rendimiento_marginal_K.png', dpi=150)
plt.show()

print('Medias por K ampliado:')
for K, m in zip(K_extra, medias_extra):
    print(f'  K={K:2d} → {m:.2f} sobres')

print()
print('  Conclusión: la curva es decreciente y cóncava. La mayor ganancia')
print('  se da entre K=10 y K=5. A partir de K≤2 la mejora adicional es')
print('  pequeña, sugiriendo rendimientos marginales decrecientes.')
print('  Razón posible: cuando K es muy bajo casi cada sobre trae al menos')
print('  un canje, por lo que el beneficio marginal de reducir K más se')
print('  agota rápidamente.')

#### Pregunta 5 — Costo efectivo por estampa nueva vía canje

In [ ]:
# Costo por estampa nueva vía canje:
# Se necesitan K repetidas → esas repetidas provienen de sobres ya pagados.
# Cada estampa (en promedio) cuesta precio_sobre / S al comprarse.
# Costo efectivo del canje = K × (precio_sobre / S)

print('── Pregunta 5 — Costo efectivo por estampa vía canje ────────')
print(f'  Precio por sobre            : Q{PRECIO_SOBRE:.2f}')
print(f'  Estampas por sobre          : {S}')
print(f'  Costo por estampa (promedio): Q{PRECIO_SOBRE/S:.4f}')
print()
print(f'  {"K":>4s} │ {"Costo efectivo por canje":>26s}')
print('  ─────┼───────────────────────────')
for K in [1, 2, 5, 10]:
    costo_canje = K * (PRECIO_SOBRE / S)
    print(f'  {K:>4d} │  Q{costo_canje:6.4f} por estampa nueva')

print()
print('  La tasa más rentable es K=1 (costo mínimo por estampa vía canje).')
print('  Sin embargo, en la práctica K=1 implica canjear cada repetida')
print('  inmediatamente, lo que puede no ser factible operativamente.')

---
## Conclusiones

### Etapa 3
1. Con Q 1,000 se pueden comprar hasta **105 sobres**. A pesar de superar holgadamente el mínimo teórico de 15, la probabilidad de completar el álbum con presupuesto restringido es baja debido a las repetidas.
2. La **caja de 104 sobres (Q 975)** entrega tantos sobres como los comprables sueltos, a un costo similar; la probabilidad de éxito es comparable.
3. La **estrategia mixta** (caja + 2 sobres sueltos) maximiza los sobres dentro del presupuesto, mejorando marginalmente la probabilidad.

### Etapa 4
1. El intercambio reduce significativamente los sobres necesarios, con la mayor ganancia entre K=10 y K=5.
2. La relación no es lineal: los rendimientos son decrecientes a medida que K disminuye.
3. Con K=2 se ahorra una cantidad relevante de sobres y quetzales respecto al caso sin intercambio.
4. K=1 es la tasa más rentable por estampa nueva, aunque la mejora marginal sobre K=2 es modesta.